<a href="https://colab.research.google.com/github/NidhiDekate/car-price-multimodal-dl/blob/main/03_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

base = '/content/drive/MyDrive/CarPricePrediction/'

# load cleaned data from eda
df = pd.read_csv(base + 'data/tabular/df_cleaned.csv')
print("shape:", df.shape)
print("columns:", df.columns.tolist())

Mounted at /content/drive
shape: (40437, 17)
columns: ['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate', 'stanford_class']


In [2]:
# drop columns we decided during eda
# vin = unique id, no pattern
# mmr = data leakage (0.98 correlation with price)

# saledate = all sales happened in 2014-2015 auction period
# we already have 'year' which captures car age
# month/season effects are minor compared to make/model/odometer
# dropping to keep feature set clean for now

df = df.drop(columns=['vin', 'mmr', 'saledate'])
print("shape after dropping:", df.shape)
print("remaining columns:", df.columns.tolist())

shape after dropping: (40437, 14)
remaining columns: ['year', 'make', 'model', 'trim', 'body', 'transmission', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'sellingprice', 'stanford_class']


In [3]:
# feature engineering

# 1. car age - more intuitive than raw year
# dataset is from 2015 auctions so we use 2015 as reference
df['car_age'] = 2015 - df['year']

# 2. log odometer - makes relationship with price more linear
df['log_odometer'] = np.log1p(df['odometer'])

print("car age stats:")
print(df['car_age'].describe())
print("\nlog odometer stats:")
print(df['log_odometer'].describe())
print("\nsample - year vs car_age:")
print(df[['year', 'car_age']].head(5))

car age stats:
count    40437.000000
mean         3.706037
std          1.622551
min          3.000000
25%          3.000000
50%          3.000000
75%          3.000000
max         22.000000
Name: car_age, dtype: float64

log odometer stats:
count    40437.000000
mean        10.639832
std          0.652445
min          0.693147
25%         10.268165
50%         10.584005
75%         11.034309
max         12.205417
Name: log_odometer, dtype: float64

sample - year vs car_age:
   year  car_age
0  2012        3
1  2012        3
2  2012        3
3  2012        3
4  2012        3


### car_age
- calculated as 2015 - year (dataset is from 2015 auctions)
- more direct than raw year — model sees "3 years old" not "2012"
- note: 75% of our matched dataset is from 2012 (stanford limitation)
  so car_age has low variation — most rows will show car_age=3
  this means car_age won't be a strong feature in our model
  but we keep it since it costs nothing

### log_odometer  
- log transform of raw odometer miles
- makes the relationship with price more linear
- raw odometer has diminishing returns (difference between
  10k and 20k miles matters more than 90k vs 100k miles)
- log captures this non-linear relationship naturally

In [4]:
# brand tier - group makes into price tiers based on eda findings
luxury = ['Ferrari', 'Rolls-Royce', 'Bentley', 'Lamborghini', 'Aston Martin']
premium = ['Porsche', 'Jaguar', 'Bmw', 'Mercedes-Benz', 'Audi', 'Cadillac', 'Infiniti', 'Acura', 'Lexus']
mainstream = ['Ford', 'Chevrolet', 'Toyota', 'Honda', 'Nissan', 'Hyundai', 'Dodge',
              'Jeep', 'Kia', 'Volkswagen', 'Mazda', 'Subaru', 'Buick', 'Gmc']

def get_brand_tier(make):
    if make in luxury:
        return 2
    elif make in premium:
        return 1
    else:
        return 0

df['brand_tier'] = df['make'].apply(get_brand_tier)

print("brand tier distribution:")
print(df['brand_tier'].value_counts())
print("\nsample:")
print(df[['make', 'brand_tier']].drop_duplicates().sort_values('brand_tier', ascending=False).head(10))

brand tier distribution:
brand_tier
0    32269
1     8137
2       31
Name: count, dtype: int64

sample:
               make  brand_tier
430         Bentley           2
8987    Rolls-Royce           2
127         Ferrari           2
7              Audi           1
1               Bmw           1
0             Acura           1
79         Cadillac           1
247          Jaguar           1
326         Porsche           1
241   Mercedes-Benz           1


In [5]:
# check nulls in remaining columns
print("null counts:")
print(df.isnull().sum())

null counts:
year                 0
make                 0
model                0
trim                28
body                66
transmission      5175
state                0
condition          369
odometer             0
color               84
interior            84
seller               0
sellingprice         0
stanford_class       0
car_age              0
log_odometer         0
brand_tier           0
dtype: int64


In [6]:
# handle missing values - each decision has a reason

# transmission - 5175 nulls - fill with mode (most common value)
# reason: transmission type is important feature, mode is safest assumption
most_common_transmission = df['transmission'].mode()[0]
df['transmission'] = df['transmission'].fillna(most_common_transmission)
print("transmission filled with:", most_common_transmission)

# condition - 369 nulls - fill with median
# reason: condition is numeric, median is robust to outliers
median_condition = df['condition'].median()
df['condition'] = df['condition'].fillna(median_condition)
print("condition filled with:", median_condition)

# body - 66 nulls - fill with mode
df['body'] = df['body'].fillna(df['body'].mode()[0])

# color - 84 nulls - fill with 'unknown'
# reason: unknown color is valid information itself
df['color'] = df['color'].fillna('unknown')

# interior - 84 nulls - fill with 'unknown'
df['interior'] = df['interior'].fillna('unknown')

# trim - 28 nulls - fill with 'base'
# reason: missing trim usually means base model
df['trim'] = df['trim'].fillna('base')

print("\nnull counts after filling:")
print(df.isnull().sum())

transmission filled with: automatic
condition filled with: 36.0

null counts after filling:
year              0
make              0
model             0
trim              0
body              0
transmission      0
state             0
condition         0
odometer          0
color             0
interior          0
seller            0
sellingprice      0
stanford_class    0
car_age           0
log_odometer      0
brand_tier        0
dtype: int64


In [7]:
from sklearn.preprocessing import LabelEncoder

# columns that need encoding
cat_cols = ['make', 'model', 'trim', 'body', 'transmission',
            'state', 'color', 'interior', 'seller']

# store encoders in case we need to inverse transform later
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {df[col].nunique()} unique values → encoded")

print("\nshape after encoding:", df.shape)

make: 35 unique values → encoded
model: 140 unique values → encoded
trim: 248 unique values → encoded
body: 34 unique values → encoded
transmission: 2 unique values → encoded
state: 38 unique values → encoded
color: 19 unique values → encoded
interior: 18 unique values → encoded
seller: 3749 unique values → encoded

shape after encoding: (40437, 26)


In [8]:
# drop seller - 3749 unique values, too sparse to be useful
# most sellers appear only a handful of times
df = df.drop(columns=['seller', 'seller_encoded'])
print("dropped seller column")
print("shape:", df.shape)

dropped seller column
shape: (40437, 24)


In [9]:
# define final feature columns for tabular branch
# using encoded versions for categorical columns
feature_cols = [
    'car_age',
    'log_odometer',
    'condition',
    'brand_tier',
    'make_encoded',
    'model_encoded',
    'trim_encoded',
    'body_encoded',
    'transmission_encoded',
    'state_encoded',
    'color_encoded',
    'interior_encoded'
]

# target - log transform as decided in eda
df['log_price'] = np.log1p(df['sellingprice'])

print("features:", len(feature_cols))
print("feature list:", feature_cols)
print("\ntarget stats:")
print(df['log_price'].describe())

features: 12
feature list: ['car_age', 'log_odometer', 'condition', 'brand_tier', 'make_encoded', 'model_encoded', 'trim_encoded', 'body_encoded', 'transmission_encoded', 'state_encoded', 'color_encoded', 'interior_encoded']

target stats:
count    40437.000000
mean         9.540919
std          0.576578
min          6.908755
25%          9.305741
50%          9.575053
75%          9.947552
max         12.040614
Name: log_price, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df['log_price']

# first split off test set - 15%
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

# then split remaining into train and val
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42
)

# 0.176 of 0.85 = ~15% of total
print("train size:", len(X_train))
print("val size:", len(X_val))
print("test size:", len(X_test))
print("total:", len(X_train) + len(X_val) + len(X_test))

train size: 28321
val size: 6050
test size: 6066
total: 40437


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# fit ONLY on train, transform all three
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:", X_val_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (28321, 12)
X_val_scaled shape: (6050, 12)
X_test_scaled shape: (6066, 12)


In [12]:
import pickle

# save scaled arrays as numpy files
np.save(base + 'data/tabular/X_train.npy', X_train_scaled)
np.save(base + 'data/tabular/X_val.npy', X_val_scaled)
np.save(base + 'data/tabular/X_test.npy', X_test_scaled)

np.save(base + 'data/tabular/y_train.npy', y_train.values)
np.save(base + 'data/tabular/y_val.npy', y_val.values)
np.save(base + 'data/tabular/y_test.npy', y_test.values)

# save scaler for later use during inference
with open(base + 'data/tabular/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# save encoders too
with open(base + 'data/tabular/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# save df with stanford_class for image loading later
df_final = df.copy()
df_final['split'] = 'train'
df_final.loc[X_val.index, 'split'] = 'val'
df_final.loc[X_test.index, 'split'] = 'test'
df_final.to_csv(base + 'data/tabular/df_final.csv', index=False)

print("all files saved:")
print("- X_train, X_val, X_test (scaled numpy arrays)")
print("- y_train, y_val, y_test (log price targets)")
print("- scaler.pkl")
print("- encoders.pkl")
print("- df_final.csv (with split column)")

all files saved:
- X_train, X_val, X_test (scaled numpy arrays)
- y_train, y_val, y_test (log price targets)
- scaler.pkl
- encoders.pkl
- df_final.csv (with split column)


## preprocessing summary

features used (12): car_age, log_odometer, condition, brand_tier,
make, model, trim, body, transmission, state, color, interior

key decisions:
- dropped mmr (leakage), vin (unique id), seller (too sparse)
- log transformed target and odometer
- brand_tier engineered from make (luxury=2, premium=1, mainstream=0)
- scaler fitted on train only - prevents data leakage
- all files saved to drive for use in modeling notebooks

final splits: train=28,321 | val=6,050 | test=6,066